In [1]:
from pathlib import Path

RECETTES = {
    "omelette_courgettes.txt": """Omelette aux courgettes
Temps : 15 minutes. Pour 2 personnes. Végétarien.
Ingrédients : 4 œufs, 1 courgette, 50 g de fromage râpé, huile d'olive, sel, poivre.
Préparation : Couper la courgette en fines rondelles et la faire revenir 5 minutes dans l'huile.
Battre les œufs avec le sel et le poivre, verser sur les courgettes, ajouter le fromage.
Cuire 5 minutes à feu doux.""",

    "gratin_dauphinois.txt": """Gratin dauphinois
Temps : 1 heure 15. Pour 4 personnes. Végétarien.
Ingrédients : 1 kg de pommes de terre, 50 cl de crème liquide, 25 cl de lait, 2 gousses d'ail, muscade, sel, poivre.
Préparation : Éplucher et couper les pommes de terre en fines lamelles. Frotter le plat avec l'ail.
Chauffer le lait et la crème avec la muscade. Disposer les pommes de terre, napper du mélange.
Cuire 1 heure au four à 160 °C.""",

    "poulet_curry.txt": """Poulet au curry et lait de coco
Temps : 30 minutes. Pour 4 personnes.
Ingrédients : 600 g de blanc de poulet, 40 cl de lait de coco, 1 oignon, 2 cuillères à soupe de curry, 200 g de riz basmati.
Préparation : Faire revenir l'oignon émincé, ajouter le poulet en dés et le curry.
Verser le lait de coco et laisser mijoter 20 minutes. Servir avec le riz.""",

    "pates_carbonara.txt": """Pâtes carbonara
Temps : 20 minutes. Pour 2 personnes.
Ingrédients : 250 g de spaghetti, 150 g de guanciale ou de lardons, 3 jaunes d'œufs, 60 g de pecorino, poivre.
Préparation : Cuire les pâtes. Faire dorer le guanciale. Mélanger les jaunes avec le pecorino.
Hors du feu, mélanger les pâtes, le guanciale et la crème d'œufs. Poivrer généreusement.""",

    "salade_lentilles.txt": """Salade de lentilles
Temps : 25 minutes. Pour 4 personnes. Végétarien, sans gluten.
Ingrédients : 250 g de lentilles vertes, 1 échalote, 1 carotte, persil, 3 cuillères à soupe d'huile, 1 cuillère à soupe de vinaigre, moutarde.
Préparation : Cuire les lentilles 20 minutes dans l'eau. Les égoutter et laisser tiédir.
Ajouter l'échalote, la carotte râpée, le persil et la vinaigrette.""",

    "pancakes.txt": """Pancakes
Temps : 20 minutes. Pour 4 personnes. Végétarien. Dessert ou petit-déjeuner.
Ingrédients : 2 cups de farine, 2 œufs, 1.5 cups de lait, 2 cuillères à soupe de sucre, 1 sachet de levure.
Préparation : Mélanger la farine, le sucre et la levure. Ajouter les œufs puis le lait progressivement.
Cuire des petites louches de pâte 2 minutes de chaque côté dans une poêle chaude.""",
}

dossier = Path("recettes")
dossier.mkdir(exist_ok=True)
for nom, texte in RECETTES.items():
    (dossier / nom).write_text(texte, encoding="utf-8")

print(sorted(p.name for p in dossier.glob("*.txt")))

['gratin_dauphinois.txt', 'omelette_courgettes.txt', 'pancakes.txt', 'pates_carbonara.txt', 'poulet_curry.txt', 'salade_lentilles.txt']


In [2]:
from langchain_ollama import OllamaEmbeddings
import numpy as np

emb = OllamaEmbeddings(model="nomic-embed-text")

phrases = [
    "un plat sans viande",
    "une recette végétarienne",
    "du poulet au curry",
    "réparer un vélo",
]
vecteurs = np.array(emb.embed_documents(phrases))
print("Taille d'un vecteur :", vecteurs.shape[1])
print(vecteurs)

def similarite(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

for i, p in enumerate(phrases[1:], start=1):
    print(f"« {phrases[0]} » vs « {p} » : {similarite(vecteurs[0], vecteurs[i]):.2f}")

Taille d'un vecteur : 768
[[ 0.05212184  0.02238866 -0.14013636 ... -0.01383122 -0.02949705
  -0.00928882]
 [ 0.07475527  0.00307637 -0.16792385 ... -0.03690909 -0.0612443
   0.02576822]
 [-0.00478852  0.02738176 -0.16020349 ... -0.02665213 -0.01602873
  -0.05183398]
 [ 0.06938602  0.04408081 -0.18304347 ... -0.03145555 -0.0385106
  -0.03597594]]
« un plat sans viande » vs « une recette végétarienne » : 0.57
« un plat sans viande » vs « du poulet au curry » : 0.60
« un plat sans viande » vs « réparer un vélo » : 0.63


In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = [
    Document(page_content=p.read_text(encoding="utf-8"), metadata={"source": p.name})
    for p in sorted(Path("recettes").glob("*.txt"))
]

decoupeur = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
morceaux = decoupeur.split_documents(documents)

print(f"{len(documents)} recettes → {len(morceaux)} morceaux\n")
print(morceaux[0].page_content, "\n---\n", morceaux[0].metadata)

6 recettes → 12 morceaux

Gratin dauphinois
Temps : 1 heure 15. Pour 4 personnes. Végétarien.
Ingrédients : 1 kg de pommes de terre, 50 cl de crème liquide, 25 cl de lait, 2 gousses d'ail, muscade, sel, poivre.
Préparation : Éplucher et couper les pommes de terre en fines lamelles. Frotter le plat avec l'ail. 
---
 {'source': 'gratin_dauphinois.txt'}


In [4]:
from langchain_chroma import Chroma

base = Chroma.from_documents(morceaux, embedding=emb, collection_name="recettes")
retriever = base.as_retriever(search_kwargs={"k": 3})

for question in ["un plat végétarien rapide", "j'ai du poulet", "quelque chose de sucré"]:
    print(f"\n🔎 {question}")
    for doc in retriever.invoke(question):
        print(f"   → {doc.metadata['source']:<26} {doc.page_content[:60]!r}")


🔎 un plat végétarien rapide
   → poulet_curry.txt           'Verser le lait de coco et laisser mijoter 20 minutes. Servir'
   → salade_lentilles.txt       'Salade de lentilles\nTemps : 25 minutes. Pour 4 personnes. Vé'
   → gratin_dauphinois.txt      'Gratin dauphinois\nTemps : 1 heure 15. Pour 4 personnes. Végé'

🔎 j'ai du poulet
   → omelette_courgettes.txt    'Battre les œufs avec le sel et le poivre, verser sur les cou'
   → poulet_curry.txt           'Poulet au curry et lait de coco\nTemps : 30 minutes. Pour 4 p'
   → poulet_curry.txt           'Verser le lait de coco et laisser mijoter 20 minutes. Servir'

🔎 quelque chose de sucré
   → poulet_curry.txt           'Verser le lait de coco et laisser mijoter 20 minutes. Servir'
   → omelette_courgettes.txt    'Battre les œufs avec le sel et le poivre, verser sur les cou'
   → pancakes.txt               'Cuire des petites louches de pâte 2 minutes de chaque côté d'


In [5]:
import sys
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "cuisine": {
        "command": sys.executable,
        "args": [str(Path("serveur_mcp.py").resolve())],
        "transport": "stdio",
    }
})

outils_mcp = await client.get_tools()
for o in outils_mcp:
    print(f"🔧 {o.name} : {o.description.splitlines()[0]}")

print()
print(await outils_mcp[2].ainvoke({"quantite": 2, "unite": "cups", "ingredient": "farine"}))

🔧 ajouter_courses : Ajoute un ou plusieurs ingrédients à la liste de courses.
🔧 voir_courses : Affiche la liste de courses actuelle.
🔧 convertir_unites : Convertit une unité américaine (cups, tbsp, oz) en grammes ou millilitres.

[{'type': 'text', 'text': '2.0 cup de farine ≈ 250 g', 'id': 'lc_f9ceded4-52f6-49f3-95fa-2f1b64a9eea6'}]


In [6]:
from langchain_core.tools import tool

@tool
def chercher_recettes(requete: str) -> str:
    """Cherche dans le livre de recettes celles qui correspondent à une demande
    (ingrédients disponibles, type de plat, régime, temps de préparation).
    Renvoie le texte des recettes trouvées avec le nom du fichier source."""
    docs = retriever.invoke(requete)
    return "\n\n".join(f"[Source : {d.metadata['source']}]\n{d.page_content}" for d in docs)

print(chercher_recettes.invoke("pancakes")[:300])

[Source : pancakes.txt]
Pancakes
Temps : 20 minutes. Pour 4 personnes. Végétarien. Dessert ou petit-déjeuner.
Ingrédients : 2 cups de farine, 2 œufs, 1.5 cups de lait, 2 cuillères à soupe de sucre, 1 sachet de levure.
Préparation : Mélanger la farine, le sucre et la levure. Ajouter les œufs puis le 


In [12]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

llm = ChatOllama(model="qwen2.5:7b", temperature=0)


CONSIGNES = """Tu es un assistant de cuisine. Tu réponds en français.
- Pour toute question sur une recette, utilise TOUJOURS l'outil chercher_recettes.
  Ne propose que des recettes trouvées par cet outil, et cite le fichier source.
- Si une recette utilise des unités américaines (cups, tbsp, oz), convertis-les avec convertir_unites.
- Pour gérer la liste de courses, utilise ajouter_courses et voir_courses.
- Si l'information n'est pas dans les recettes, dis-le honnêtement."""

agent = create_agent(
    model=llm,
    tools=[chercher_recettes, *outils_mcp],
    system_prompt=CONSIGNES,
)

In [13]:
async def demander(question):
    resultat = await agent.ainvoke({"messages": [{"role": "user", "content": question}]})
    for message in resultat["messages"]:
        message.pretty_print()

await demander("Je veux un plat végétarien rapide, qu'est-ce que tu me proposes ?")

================================ Human Message =================================

Je veux un plat végétarien rapide, qu'est-ce que tu me proposes ?
================================== Ai Message ==================================
Tool Calls:
  chercher_recettes (550020ca-8586-4c3e-9f2f-f1a35aabddfa)
 Call ID: 550020ca-8586-4c3e-9f2f-f1a35aabddfa
  Args:
    requete: plat végétarien rapide
================================= Tool Message =================================
Name: chercher_recettes

[Source : salade_lentilles.txt]
Salade de lentilles
Temps : 25 minutes. Pour 4 personnes. Végétarien, sans gluten.
Ingrédients : 250 g de lentilles vertes, 1 échalote, 1 carotte, persil, 3 cuillères à soupe d'huile, 1 cuillère à soupe de vinaigre, moutarde.

[Source : omelette_courgettes.txt]
Omelette aux courgettes
Temps : 15 minutes. Pour 2 personnes. Végétarien.
Ingrédients : 4 œufs, 1 courgette, 50 g de fromage râpé, huile d'olive, sel, poivre.
Préparation : Couper la courgette en fines rondell

In [14]:
await demander("Je veux faire des pancakes. Donne-moi les quantités en grammes, "
               "puis ajoute la farine et le lait à ma liste de courses.")

================================ Human Message =================================

Je veux faire des pancakes. Donne-moi les quantités en grammes, puis ajoute la farine et le lait à ma liste de courses.
================================== Ai Message ==================================
Tool Calls:
  chercher_recettes (b2b0dc1f-6157-4449-b1ee-564b9d66d9be)
 Call ID: b2b0dc1f-6157-4449-b1ee-564b9d66d9be
  Args:
    requete: pancakes
================================= Tool Message =================================
Name: chercher_recettes

[Source : pancakes.txt]
Pancakes
Temps : 20 minutes. Pour 4 personnes. Végétarien. Dessert ou petit-déjeuner.
Ingrédients : 2 cups de farine, 2 œufs, 1.5 cups de lait, 2 cuillères à soupe de sucre, 1 sachet de levure.
Préparation : Mélanger la farine, le sucre et la levure. Ajouter les œufs puis le lait progressivement.

[Source : poulet_curry.txt]
Poulet au curry et lait de coco
Temps : 30 minutes. Pour 4 personnes.
Ingrédients : 600 g de blanc de poulet, 40

In [15]:
await demander("Qu'est-ce qu'il y a sur ma liste de courses ?")

================================ Human Message =================================

Qu'est-ce qu'il y a sur ma liste de courses ?
================================== Ai Message ==================================
Tool Calls:
  voir_courses (f588a33a-c9f0-49f7-9efe-d28a6078c13e)
 Call ID: f588a33a-c9f0-49f7-9efe-d28a6078c13e
  Args:
================================= Tool Message =================================
Name: voir_courses

[{'type': 'text', 'text': '- farine\n- lait', 'id': 'lc_e7de2928-3f31-41f5-9877-e6cd571aae66'}]
================================== Ai Message ==================================

Voici ce qui se trouve sur votre liste de courses : farine et lait.


In [23]:
%pip install grandalf
import sys
print(sys.executable)


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
/Users/onizukalto/ml/bin/python


In [24]:
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

outils = [chercher_recettes, *outils_mcp]
llm_avec_outils = llm.bind_tools(outils)

# Nœud 1 : le modèle réfléchit
async def modele(state: MessagesState):
    messages = [SystemMessage(CONSIGNES)] + state["messages"]
    reponse = await llm_avec_outils.ainvoke(messages)
    return {"messages": [reponse]}

# Assemblage du graphe
graphe = StateGraph(MessagesState)
graphe.add_node("modele", modele)
graphe.add_node("outils", ToolNode(outils))

graphe.add_edge(START, "modele")
graphe.add_conditional_edges("modele", tools_condition, {"tools": "outils", END: END})
graphe.add_edge("outils", "modele")

mon_agent = graphe.compile()
print(mon_agent.get_graph().draw_mermaid())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	modele(modele)
	outils(outils)
	__end__([<p>__end__</p>]):::last
	__start__ --> modele;
	modele -.-> __end__;
	modele -. &nbsp;tools&nbsp; .-> outils;
	outils --> modele;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [25]:
resultat = await mon_agent.ainvoke({"messages": [{"role": "user", "content": "Qu'est-ce qu'il y a sur ma liste de courses ?"}]})
for m in resultat["messages"]:
    m.pretty_print()

================================ Human Message =================================

Qu'est-ce qu'il y a sur ma liste de courses ?
================================== Ai Message ==================================
Tool Calls:
  voir_courses (1d529592-e2e3-4b33-8a87-feb5a6362d35)
 Call ID: 1d529592-e2e3-4b33-8a87-feb5a6362d35
  Args:
================================= Tool Message =================================
Name: voir_courses

[{'type': 'text', 'text': '- farine\n- lait', 'id': 'lc_7464032a-21bf-4ba5-8194-169dad2599f9'}]
================================== Ai Message ==================================

Voici ce qui se trouve sur votre liste de courses : farine et lait.
